# 01 Build SFT Dataset (9B Base, 28 labels)

简洁版，严格对齐当前原始数据 schema：

- 正样本来自 `record["positive_sample"]`
- 负样本来自 `record["hard_negatives"]`
- 先按 **record** 切分，再展开样本
- 输出：
  - `qwen_sft_train.jsonl`
  - `qwen_sft_dev.jsonl`
  - `qwen_sft_test.jsonl`
  - `test_ground_truth.jsonl`
  - `invalid_target_spans_top100.json`
  - `type_coverage_report.json`
  - `meta.json`


In [1]:

import json, os, random, sys
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, ".")
from redaction_utils import annotate_text


In [2]:

RAW_DATA_PATH = "../data/raw/au_pii_19000_final.json"
SAVE_DIR = "../data/processed"
SEED = 42

TRAIN_RATIO = 0.8
DEV_RATIO = 0.1
TEST_RATIO = 0.1
NEG_POS_RATIO = 1.0
EXCLUDE_EMPTY_POSITIVES = False

TARGET_LABELS = [
    "PERSON","ADDRESS","DATE_OF_BIRTH","EMAIL_ADDRESS","AU_PHONE","AU_TFN",
    "AU_PASSPORT","AU_DRIVERS_LICENCE","STUDENT_ID","MEDICARE_NUMBER",
    "AU_BANK_ACCOUNT","BSB","PAYMENT_CARD_NUMBER","IP_ADDRESS","VEHICLE_REGO","SALARY",
    "WORK_EMAIL","WORK_PHONE","EMPLOYEE_NUMBER","PERSONNEL_NUMBER","MEDICARE_EXPIRY",
    "PASSPORT_EXPIRY","UAC_ID","USI","CENTRELINK_REFERENCE_NUMBER","AU_BANK_NAME",
    "CREDIT_CARD_EXPIRY","CREDIT_CARD_CVV",
]
REQUESTED_TARGET_LABELS = list(TARGET_LABELS)
TARGET_SET = set(TARGET_LABELS)

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
random.seed(SEED)

print("RAW_DATA_PATH =", RAW_DATA_PATH)
print("SAVE_DIR      =", SAVE_DIR)
print("TARGET_LABELS =", len(TARGET_LABELS))


RAW_DATA_PATH = ../data/raw/au_pii_19000_final.json
SAVE_DIR      = ../data/processed
TARGET_LABELS = 28


In [3]:

with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

records = raw["records"]

source_target_counts = Counter()
for rec in records:
    for lab in rec["positive_sample"].get("labels", []):
        if lab["type"] in TARGET_SET:
            source_target_counts[lab["type"]] += 1

zero_positive_labels = [t for t in REQUESTED_TARGET_LABELS if source_target_counts[t] == 0]
if zero_positive_labels:
    TARGET_LABELS = [t for t in TARGET_LABELS if source_target_counts[t] > 0]
    TARGET_SET = set(TARGET_LABELS)

print("version       =", raw.get("version"))
print("total_records =", len(records))
print("source pii_types =", len(raw.get("pii_types", [])))
print("dropped zero-positive target labels =", zero_positive_labels)
print("effective target label count =", len(TARGET_LABELS))


version       = 2.0-final
total_records = 19000
source pii_types = 77
dropped zero-positive target labels = ['AU_BANK_NAME']
effective target label count = 27


In [4]:

SYSTEM_PROMPT = (
    "You are a PII annotator for Australian context.\n"
    "Return the SAME text with supported PII wrapped as <pii type=\"TYPE\">VALUE</pii>.\n"
    "Preserve every character exactly. Do not paraphrase, summarize, or explain.\n"
    "Wrap every occurrence of supported PII. Do not deduplicate.\n"
    "If no supported PII is present, return the input unchanged.\n"
    "Supported types:\n- " + "\n- ".join(TARGET_LABELS)
)


In [5]:

def inspect_valid_labels(text, labels):
    valid, invalid = [], []
    for lab in labels:
        start, end = lab["start"], lab["end"]
        if start < 0 or end > len(text) or start >= end:
            invalid.append({"reason": "offset_out_of_range", "label": lab})
            continue
        if text[start:end] != lab["value"]:
            invalid.append({
                "reason": "value_text_mismatch",
                "label": lab,
                "text_slice": text[start:end],
            })
            continue
        valid.append(lab)
    return valid, invalid

def make_messages(text, annotated_text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
        {"role": "assistant", "content": annotated_text},
    ]

def build_positive_sample(record_id, pos):
    text = pos["text"]
    labels = pos.get("labels", [])

    in_target = [x for x in labels if x["type"] in TARGET_SET]
    out_of_target = [x for x in labels if x["type"] not in TARGET_SET]
    valid_in_target, invalid_in_target = inspect_valid_labels(text, in_target)

    annotated_text, used_labels = annotate_text(text, valid_in_target, target_types=TARGET_SET)
    sample = {
        "id": record_id,
        "text": text,
        "annotated_text": annotated_text,
        "used_labels": used_labels,
        "messages": make_messages(text, annotated_text),
    }
    stats = {
        "out_of_target": out_of_target,
        "invalid_in_target": invalid_in_target,
        "overlaps_dropped": len(valid_in_target) - len(used_labels),
    }
    return sample, stats

def build_negative_sample(record_id, neg_idx, neg):
    text = neg["text"]
    sample = {
        "id": f"{record_id}::neg::{neg_idx}",
        "text": text,
        "annotated_text": text,
        "used_labels": [],
        "messages": make_messages(text, text),
    }
    return sample


In [6]:

prepared_records = []
type_counter = Counter()
out_of_target_counter = Counter()
invalid_target_counter = Counter()
invalid_examples = []
overlaps_dropped_total = 0
pos_empty_after_filter = 0

for i, rec in enumerate(records):
    record_id = rec.get("id", f"record-{i:05d}")
    pos_sample, stats = build_positive_sample(record_id, rec["positive_sample"])

    for lab in pos_sample["used_labels"]:
        type_counter[lab["type"]] += 1
    for lab in stats["out_of_target"]:
        out_of_target_counter[lab["type"]] += 1
    for bad in stats["invalid_in_target"]:
        invalid_target_counter[bad["label"]["type"]] += 1
        if len(invalid_examples) < 100:
            invalid_examples.append({
                "record_id": record_id,
                "reason": bad["reason"],
                "label": bad["label"],
                "text_slice": bad.get("text_slice"),
                "text_preview": pos_sample["text"][:500],
            })

    overlaps_dropped_total += stats["overlaps_dropped"]
    pos_empty_after_filter += int(len(pos_sample["used_labels"]) == 0)

    neg_samples = [
        build_negative_sample(record_id, j, neg)
        for j, neg in enumerate(rec.get("hard_negatives", []))
    ]

    prepared_records.append({
        "record_id": record_id,
        "positive_sample": pos_sample,
        "hard_negative_samples": neg_samples,
        "positive_empty_after_filter": len(pos_sample["used_labels"]) == 0,
    })

if EXCLUDE_EMPTY_POSITIVES:
    prepared_records = [r for r in prepared_records if not r["positive_empty_after_filter"]]

print("prepared_records        =", len(prepared_records))
print("pos_empty_after_filter  =", pos_empty_after_filter)
print("overlaps_dropped_total  =", overlaps_dropped_total)
print("invalid_target_total    =", sum(invalid_target_counter.values()))
print("top out_of_target       =", out_of_target_counter.most_common(12))


prepared_records        = 19000
pos_empty_after_filter  = 0
overlaps_dropped_total  = 0
invalid_target_total    = 0
top out_of_target       = [('IHI', 1500), ('WAM_SCORE', 1500), ('SANCTIONS', 1500), ('SCHOLARSHIP', 1500), ('SUBJECT_RESULTS', 1500), ('EMPLOYMENT_INFORMATION', 1200), ('CONTRACT_TYPE', 1200), ('MEDICAL_INFORMATION', 1200), ('MEDICAL_CERTIFICATE', 1200), ('DISABILITY_OR_SPECIFIC_CONDITION', 1200), ('COUNSELLING_RECORDS', 1200), ('SPECIAL_CONSIDERATION', 1200)]


In [7]:

def split_records_three_way(items, seed=SEED):
    train_records, temp_records = train_test_split(
        items, test_size=(1 - TRAIN_RATIO), random_state=seed, shuffle=True
    )
    dev_ratio_within_temp = DEV_RATIO / (DEV_RATIO + TEST_RATIO)
    dev_records, test_records = train_test_split(
        temp_records, test_size=(1 - dev_ratio_within_temp), random_state=seed, shuffle=True
    )
    return train_records, dev_records, test_records

def expand_split(records_split, neg_pos_ratio=NEG_POS_RATIO, seed=SEED, keep_all_negatives=False):
    positives = [r["positive_sample"] for r in records_split]
    negatives = []
    for r in records_split:
        negatives.extend(r["hard_negative_samples"])
    rng = random.Random(seed + len(records_split))
    rng.shuffle(negatives)
    if keep_all_negatives:
        kept_negatives = negatives
    else:
        n_keep = min(len(negatives), int(len(positives) * neg_pos_ratio))
        kept_negatives = negatives[:n_keep]
    all_samples = positives + kept_negatives
    rng.shuffle(all_samples)
    return all_samples, positives, kept_negatives

train_records, dev_records, test_records = split_records_three_way(prepared_records)
train_data, pos_tr, neg_tr = expand_split(train_records, seed=SEED + 1)
dev_data, pos_dv, neg_dv = expand_split(dev_records, seed=SEED + 2, keep_all_negatives=True)
test_data, pos_te, neg_te = expand_split(test_records, seed=SEED + 3, keep_all_negatives=True)

print("record split:", len(train_records), len(dev_records), len(test_records))
print("sample split:", len(train_data), len(dev_data), len(test_data))
print("train pos/neg:", len(pos_tr), len(neg_tr))
print("dev pos/neg:", len(pos_dv), len(neg_dv))
print("test pos/neg:", len(pos_te), len(neg_te))


record split: 15200 1900 1900
sample split: 30400 9495 9460
train pos/neg: 15200 15200
dev pos/neg: 1900 7595
test pos/neg: 1900 7560


In [8]:

def count_types(samples):
    c = Counter()
    for s in samples:
        for lab in s.get("used_labels", []):
            c[lab["type"]] += 1
    return c

train_type_counts = count_types(train_data)
dev_type_counts = count_types(dev_data)
test_type_counts = count_types(test_data)

type_coverage_report = []
for t in TARGET_LABELS:
    type_coverage_report.append({
        "type": t,
        "train": train_type_counts[t],
        "dev": dev_type_counts[t],
        "test": test_type_counts[t],
        "total": train_type_counts[t] + dev_type_counts[t] + test_type_counts[t],
    })

type_coverage_report[:5]


[{'type': 'PERSON', 'train': 15200, 'dev': 1900, 'test': 1900, 'total': 19000},
 {'type': 'ADDRESS', 'train': 9251, 'dev': 1168, 'test': 1163, 'total': 11582},
 {'type': 'DATE_OF_BIRTH',
  'train': 8651,
  'dev': 1079,
  'test': 1087,
  'total': 10817},
 {'type': 'EMAIL_ADDRESS',
  'train': 4089,
  'dev': 507,
  'test': 550,
  'total': 5146},
 {'type': 'AU_PHONE', 'train': 4034, 'dev': 505, 'test': 532, 'total': 5071}]

In [9]:

def save_jsonl(path, items):
    with open(path, "w", encoding="utf-8") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"saved: {path} ({len(items)})")

# test ground truth: 对 test_data 中每个样本都保留 text / annotated_text / labels
test_gt = [
    {
        "id": s["id"],
        "text": s["text"],
        "annotated_text": s["annotated_text"],
        "labels": s.get("used_labels", []),
        "messages": s["messages"],
    }
    for s in test_data
]

save_jsonl(os.path.join(SAVE_DIR, "qwen_sft_train.jsonl"), train_data)
save_jsonl(os.path.join(SAVE_DIR, "qwen_sft_dev.jsonl"), dev_data)
save_jsonl(os.path.join(SAVE_DIR, "qwen_sft_test.jsonl"), test_data)
save_jsonl(os.path.join(SAVE_DIR, "test_ground_truth.jsonl"), test_gt)

with open(os.path.join(SAVE_DIR, "invalid_target_spans_top100.json"), "w", encoding="utf-8") as f:
    json.dump(invalid_examples, f, ensure_ascii=False, indent=2)

with open(os.path.join(SAVE_DIR, "type_coverage_report.json"), "w", encoding="utf-8") as f:
    json.dump(type_coverage_report, f, ensure_ascii=False, indent=2)

meta = {
    "schema": "span-tagged",
    "tag_format": '<pii type="TYPE">VALUE</pii>',
    "source_version": raw.get("version"),
    "requested_target_labels": REQUESTED_TARGET_LABELS,
    "target_labels": TARGET_LABELS,
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "dev_ratio": DEV_RATIO,
    "test_ratio": TEST_RATIO,
    "neg_pos_ratio": NEG_POS_RATIO,
    "dev_test_keep_all_negatives": True,
    "record_count": len(prepared_records),
    "train_record_count": len(train_records),
    "dev_record_count": len(dev_records),
    "test_record_count": len(test_records),
    "train_size": len(train_data),
    "dev_size": len(dev_data),
    "test_size": len(test_data),
    "type_counts_total": dict(type_counter),
    "train_type_counts": dict(train_type_counts),
    "dev_type_counts": dict(dev_type_counts),
    "test_type_counts": dict(test_type_counts),
    "invalid_target_counts": dict(invalid_target_counter),
    "out_of_target_top20": dict(out_of_target_counter.most_common(20)),
    "overlaps_dropped_total": overlaps_dropped_total,
    "pos_empty_after_filter": pos_empty_after_filter,
    "excluded_empty_positives": EXCLUDE_EMPTY_POSITIVES,
    "zero_positive_labels_dropped": zero_positive_labels,
}
with open(os.path.join(SAVE_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("saved extra diagnostics and meta.json")


saved: ../data/processed/qwen_sft_train.jsonl (30400)
saved: ../data/processed/qwen_sft_dev.jsonl (9495)
saved: ../data/processed/qwen_sft_test.jsonl (9460)
saved: ../data/processed/test_ground_truth.jsonl (9460)
saved extra diagnostics and meta.json


In [10]:

# 简单 sanity check
print(train_data[0]["messages"][0]["content"][:200], "...")
print()
print("USER:")
print(train_data[0]["messages"][1]["content"][:400])
print()
print("ASSISTANT:")
print(train_data[0]["messages"][2]["content"][:600])


You are a PII annotator for Australian context.
Return the SAME text with supported PII wrapped as <pii type="TYPE">VALUE</pii>.
Preserve every character exactly. Do not paraphrase, summarize, or expl ...

USER:
Gym access fob 1811393510536221 was deactivated after the membership lapsed.

ASSISTANT:
Gym access fob 1811393510536221 was deactivated after the membership lapsed.
